In [ ]:
%pip install openai scikit-learn pandas

In [ ]:
# Create client

from openai import OpenAI
from sklearn.model_selection import train_test_split
import pandas as pd

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",  # Ollama
    api_key="sk-1234"  # Dummy key
)

In [ ]:
SYSTEM_PROMPT = """
You are a GAD-7 survey scoring assistant.
Your job is to extract answers for the GAD-7 based on a given conversation transcript.
The Generalized Anxiety Disorder 7-item scale (GAD-7) is a clinical assessment tool that measures the severity of anxiety symptoms over the past two weeks.

The conversation asks questions in this specific order. Map each user response to the correct q number:
q1   Feeling nervous, anxious, or on edge
q2   Not being able to stop or control worrying
q3   Worrying too much about different things
q4   Trouble relaxing
q5   Being so restless that it's hard to sit still
q6   Becoming easily annoyed or irritable
q7   Feeling afraid, as if something awful might happen

After q7, the assistant asks a follow-up about how much these feelings have interfered with daily life (work, home, relationships). This is the functional impairment question and is NOT scored 0-3 — it is recorded as one of these four exact strings:
  "Not difficult at all"
  "Somewhat difficult"
  "Very difficult"
  "Extremely difficult"

Scale mapping for q1-q7:
0 = not at all
1 = several days
2 = more than half the days
3 = nearly every day

Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
- For q1 through q7, return a single string value: "0", "1", "2", or "3".
- For functional_impairment, return one of the four exact strings listed above.
Rules:
- Match each topic to the correct q number based on the descriptions above. The interviewer may ask multiple follow-up questions for the same item; treat the user's overall answer for that topic as the value.
- Map natural-language frequencies to the closest scale value (e.g., "not at all" / "never" -> 0, "a few days" / "once or twice" / "several days" -> 1, "more than half the days" / "most days" -> 2, "every day" / "nearly every day" / "constantly" / "all the time" -> 3).
- Map the final functional impairment answer to the closest of the four labels (e.g., a user saying "it's been quite a bit, very difficult" -> "Very difficult"; "barely affects me" -> "Not difficult at all"; "I'm drowning" / "completely overwhelmed" -> "Extremely difficult").
- Score each item based solely on what the user said.
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any key.
Return JSON in exactly this shape:
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0",
  "functional_impairment": "Not difficult at all"
}
"""
MODEL = 'qwen3:8b'

- Use a json dataset of GAD-7 conversation transcripts
- Extract scores using LLM
- Validate whether they were correct using MSE on q1-q7 and accuracy on functional_impairment

In [ ]:
# Load the GAD-7 dataset
import json

with open('gad7_valid_dataset.json') as f:
    data = json.loads('\n'.join([line for line in f]))

# Split the data into conversation data and expected values
conversations = [entry['conversation'] for entry in data]
expected_outputs = [entry['expected_output'] for entry in data]

print('SUCCESS: Number of conversations matches number of scores' if len(conversations) == len(expected_outputs) else 'FAIL: Number of conversations does not match number of scores')
print(f'Total conversations: {len(conversations)}')

In [ ]:
# Further split the data into training and test data
conv_train, _conv_test, score_train, _score_test = train_test_split(conversations, expected_outputs, random_state=1234)
print(f'Train: {len(conv_train)}, Test: {len(_conv_test)}')

In [ ]:
def Message(content, role='user'):
    return {
        'role': role,
        'content': content
    }

def create_prediction(transcript) -> str:
    completion = client.chat.completions.create(
        model=MODEL,
        temperature=0.1,  # low temperature for consistent scoring
        messages=[
            Message(SYSTEM_PROMPT, 'system'),
            Message(json.dumps(transcript))
        ],
    )
    return completion.choices[0].message.content

In [ ]:
# Run predictions on training set
scores = []
for i, c in enumerate(conv_train):
    print(f'Generating {i+1}/{len(conv_train)}')
    scores.append(create_prediction(c))
    print(scores[-1])

In [ ]:
# Parse JSON scores
json_scores = []
default = {f'q{i}': '0' for i in range(1, 8)}
default['functional_impairment'] = 'Not difficult at all'
for s in scores:
    try:
        json_scores.append(json.loads(s))
    except json.JSONDecodeError:
        print(f'Failed to parse: {s}')
        json_scores.append(dict(default))

In [ ]:
from sklearn.metrics import mean_squared_error

def convert_scores_to_array(scores_df):
    return [int(s) if s else 0 for s in scores_df.values.flatten().tolist()]

pred_df = pd.DataFrame(json_scores)
train_df = pd.DataFrame(score_train)

# Numeric items q1-q7 for MSE
num_cols = [f'q{i}' for i in range(1, 8)]
pred_num = pred_df[num_cols]
train_num = train_df[num_cols]

mse = mean_squared_error(
    convert_scores_to_array(train_num),
    convert_scores_to_array(pred_num)
)
print(f'MSE (q1-q7): {mse}')

# Functional impairment accuracy
fi_correct = (pred_df['functional_impairment'] == train_df['functional_impairment']).sum()
print(f'Functional impairment accuracy: {fi_correct}/{len(train_df)}')

# Combined diff view
all_cols = num_cols + ['functional_impairment']
diff_df = train_df[all_cols].compare(pred_df[all_cols], align_axis=1, keep_shape=True, keep_equal=True).rename(
    columns={'self': 'Expected', 'other': 'Predicted'}, level=1
)
display(diff_df)